# Dev-only GPU smoke test (not production)

Runs the FastAPI backend (`server/`) on Colab's free T4 GPU and exposes it via an ngrok tunnel, so we can test the LLM/Whisper/TTS/moderation pipeline actually works end-to-end before committing to paid GPU hosting.

**This is explicitly a throwaway dev harness, not the production backend** — free Colab sessions disconnect after ~90min idle / ~12hr max, and the ngrok URL changes every time you reconnect. Do not point real users at this.

Before running: **Runtime -> Change runtime type -> T4 GPU**.

You'll need a free ngrok account + authtoken from https://dashboard.ngrok.com/get-started/your-authtoken (Cell 6 will prompt for it, input is hidden).

In [ ]:
!nvidia-smi

In [ ]:
import os
if not os.path.exists('/content/SinfulSparks'):
    !git clone https://github.com/siliconcode-dev/SinfulSparks.git /content/SinfulSparks
else:
    !cd /content/SinfulSparks && git pull
%cd /content/SinfulSparks/server

In [ ]:
# Skips torch/torchvision/torchaudio — Colab already ships a CUDA-matched
# torch build, and reinstalling from requirements.txt risks breaking that.
!pip install -q fastapi uvicorn python-multipart pydantic python-dotenv \
  supabase transformers peft accelerate bitsandbytes faster-whisper TTS pyngrok nest_asyncio

In [ ]:
import os

# Small/fast overrides for this smoke test only — production (Cloud Run)
# keeps the real Qwen2.5-14B + Whisper-medium defaults (see server/app/llm.py,
# server/app/stt.py). The 14B model alone is a ~28GB download, impractical
# on a free Colab session — this is about validating the pipeline wiring,
# not final dialogue quality.
os.environ['LLM_MODEL_NAME'] = 'Qwen/Qwen2.5-1.5B-Instruct'
os.environ['WHISPER_MODEL_SIZE'] = 'small'
os.environ['ALLOWED_ORIGIN'] = '*'  # dev tunnel only — never do this in production

# Deliberately NOT setting SUPABASE_URL/SUPABASE_SERVICE_ROLE_KEY here:
# unauthenticated requests are treated as anonymous by server/app/auth.py,
# which is all this smoke test needs. Don't paste real Supabase secrets into
# a Colab session for a throwaway test.

In [ ]:
import subprocess, sys, time

log_file = open('/content/server.log', 'w')
proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8080'],
    cwd='/content/SinfulSparks/server',
    stdout=log_file, stderr=subprocess.STDOUT,
)
print(f'Started server, PID={proc.pid}')
time.sleep(5)
!tail -n 30 /content/server.log

In [ ]:
from getpass import getpass
from pyngrok import ngrok, conf

conf.get_default().auth_token = getpass('ngrok authtoken: ')
public_url = ngrok.connect(8080, 'http')
print(f'Backend reachable at: {public_url}')

In [ ]:
# Health check
!curl -s {public_url}/api/health

In [ ]:
# First real dialogue call — triggers model loading (slow the first time:
# downloads + loads the LLM onto the GPU). Watch /content/server.log if this
# hangs, via: !tail -n 50 /content/server.log
import requests

resp = requests.post(f'{public_url}/api/dialogue', json={
    'characterId': 'maya',
    'message': "Hey, I like your energy today.",
    'history': [],
})
print(resp.status_code, resp.json())

## Using this from the local client

Copy the `public_url` printed above into `client/.env`:

```
VITE_BACKEND_URL=<public_url from above>
```

then `npm run dev` in `client/` and play through a conversation for real — this exercises STT (`/api/stt`), dialogue (`/api/dialogue`), and TTS (`/api/tts`) together.

Re-run the ngrok cell if it disconnects — the URL changes each time, so update `client/.env` again after.

In [ ]:
# Cleanup when done — stops the tunnel and the server (also just closing/
# disconnecting the Colab runtime achieves the same thing).
ngrok.disconnect(public_url)
proc.terminate()